# Laboratorio: Agente RAG con LangChain y OCI Vector Store

En este laboratorio construiremos un agente con **LangChain** capaz de consultar documentos almacenados en un **OCI Generative AI Vector Store**.

## Objetivos

Al finalizar este laboratorio podrás:

- Descubrir y seleccionar documentos desde una carpeta de datos.
- Conectarte a los endpoints compatibles con OpenAI de OCI Generative AI.
- Seleccionar un vector store existente sin fijar su identificador en el código.
- Cargar e indexar un documento de manera opcional.
- Ejecutar una búsqueda semántica directa.
- Convertir la búsqueda en una tool de LangChain.
- Crear un agente que responda utilizando la base de conocimiento.

---

## 1. Preparación del entorno

El laboratorio utiliza autenticación IAM de OCI y carga la configuración desde `.env`. El entorno de Python debe tener instalados `langchain`, `langchain-openai`, `openai`, `oci-openai`, `httpx` y `python-dotenv`.

El archivo `.env` debe contener como mínimo:

```text
OCI_COMPARTMENT_ID=<compartment-ocid>
OCI_GENERATIVE_AI_PROJECT_ID=<generative-ai-project-ocid>
```

También puedes definir `OCI_REGION`, `OCI_PROFILE`, `OCI_MODEL_ID`, `OCI_VECTOR_STORE_ID` y `OCI_SOURCE_DOCUMENT` para cambiar el comportamiento sin editar las celdas.

In [1]:
import os
import time
from pathlib import Path

import httpx
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from oci_openai import OciOpenAI, OciUserPrincipalAuth

### Configuración general

Cargamos las variables de entorno y construimos los endpoints. El proyecto, el vector store y los endpoints deben pertenecer a la misma región.

In [2]:
load_dotenv()

REGION = os.getenv("OCI_REGION", "us-ashburn-1")
PROFILE = os.getenv("OCI_PROFILE", "latinoamerica")
MODEL_ID = os.getenv("OCI_MODEL_ID", "openai.gpt-5")
COMPARTMENT_ID = os.environ["OCI_COMPARTMENT_ID"]
PROJECT_ID = os.environ["OCI_GENERATIVE_AI_PROJECT_ID"]
DATA_DIRECTORY = Path("data")
BASE_URL = f"https://inference.generativeai.{REGION}.oci.oraclecloud.com/openai/v1"
CONTROL_PLANE_URL = (
    f"https://generativeai.{REGION}.oci.oraclecloud.com/20231130/openai/v1"
)

---

## 2. Selección del documento

El notebook descubre automáticamente los archivos `.md` dentro de `data/`. Si `OCI_SOURCE_DOCUMENT` no está definida, selecciona el primer documento disponible.

> **Sugerencia para el laboratorio en vivo:** agrega nuevos archivos a `data/` y cambia `OCI_SOURCE_DOCUMENT` para elegir cuál cargar.

In [3]:
available_documents = sorted(DATA_DIRECTORY.glob("*.md"))
if not available_documents:
    raise FileNotFoundError("No Markdown documents were found in the data directory.")

configured_document = os.getenv("OCI_SOURCE_DOCUMENT")
document_path = (
    DATA_DIRECTORY / configured_document
    if configured_document
    else available_documents[0]
)
if not document_path.is_file():
    raise FileNotFoundError(f"Document not found: {document_path}")

print("Available documents:")
for path in available_documents:
    print(f"- {path.name}")
print(f"Selected document: {document_path.name}")

Available documents:
- lumina_credito_global.md
Selected document: lumina_credito_global.md


---

## 3. Creación de los clientes de OCI

OCI separa las operaciones de inferencia y control plane:

- `client` se utiliza para archivos, búsquedas e inferencia.
- `control_plane_client` se utiliza para administrar vector stores.

Ambos clientes utilizan el perfil IAM configurado en `PROFILE`.

In [4]:
def create_auth():
    return OciUserPrincipalAuth(profile_name=PROFILE)


client = OciOpenAI(
    auth=create_auth(),
    region=REGION,
    compartment_id=COMPARTMENT_ID,
    project=PROJECT_ID,
)

control_plane_client = OciOpenAI(
    auth=create_auth(),
    base_url=CONTROL_PLANE_URL,
    compartment_id=COMPARTMENT_ID,
    project=PROJECT_ID,
)

---

## 4. Selección del vector store

Listamos los vector stores disponibles en el compartment. Si `OCI_VECTOR_STORE_ID` está definida, utilizamos ese recurso; de lo contrario seleccionamos el más reciente.

> **Importante:** en un entorno con varios recursos conviene definir explícitamente `OCI_VECTOR_STORE_ID`.

In [5]:
vector_stores = control_plane_client.vector_stores.list(
    limit=20,
    order="desc",
)
if not vector_stores.data:
    raise RuntimeError("No vector stores are available in the configured compartment.")

configured_vector_store_id = os.getenv("OCI_VECTOR_STORE_ID")
VECTOR_STORE_ID = configured_vector_store_id or vector_stores.data[0].id
vector_store = control_plane_client.vector_stores.retrieve(
    vector_store_id=VECTOR_STORE_ID
)

print(f"Selected vector store: {vector_store.name} ({vector_store.id})")
print(f"Status: {vector_store.status}")
print(f"Indexed files: {vector_store.file_counts.completed}")

Selected vector store: vs (vs_iad_j37gxxf6hibtcqwzeuzau3nigs2fwe49dp7ahsik5yqyu4pz)
Status: completed
Indexed files: 4


---

## 5. Carga opcional del documento

La carga está desactivada de manera predeterminada para evitar duplicados al ejecutar el notebook varias veces. Para cargar el documento seleccionado, cambia `UPLOAD_DOCUMENT` a `True` o define `OCI_UPLOAD_DOCUMENT=true`.

El proceso crea un archivo, lo agrega al vector store mediante un batch y espera hasta que la indexación termine.

In [6]:
UPLOAD_DOCUMENT = os.getenv("OCI_UPLOAD_DOCUMENT", "false").lower() == "true"

if UPLOAD_DOCUMENT:
    with document_path.open("rb") as file_content:
        uploaded_file = client.files.create(
            file=file_content,
            purpose="user_data",
        )

    batch = client.vector_stores.file_batches.create(
        vector_store_id=VECTOR_STORE_ID,
        file_ids=[uploaded_file.id],
        attributes={"source": document_path.name, "format": "markdown"},
    )

    while True:
        batch_status = client.vector_stores.file_batches.retrieve(
            vector_store_id=VECTOR_STORE_ID,
            batch_id=batch.id,
        )
        if batch_status.status in {"completed", "failed", "cancelled"}:
            break
        time.sleep(2)

    print(f"File: {uploaded_file.id}")
    print(f"Batch: {batch.id}")
    print(f"Status: {batch_status.status}")
else:
    print("Document upload skipped.")

Document upload skipped.


---

## 6. Búsqueda directa en la base de conocimiento

Probamos la búsqueda sin utilizar LangChain. La pregunta está en español, pero el código y los nombres de variables permanecen en inglés.

> **Nota:** `rewrite_query` se establece en `False` porque esta opción no está soportada por el endpoint utilizado.

In [7]:
search_results = client.vector_stores.search(
    vector_store_id=VECTOR_STORE_ID,
    query="¿Cuáles son los temas y condiciones más importantes del documento?",
    max_num_results=3,
    rewrite_query=False,
)

for result in search_results.data:
    print(f"File: {result.filename}")
    print(f"Relevance: {result.score:.3f}")
    print(result.content[0].text[:500])
    print("-" * 80)

File: lumina_credito_global.md
Relevance: 0.100
lumina_credito_global.md: Lúmina Crédito Global: 16. Canales ficticios de atención

- Sitio web: `https://www.luminacredito.example`
- Correo general: `ayuda@luminacredito.example`
- Correo para pagos: `pagos@luminacredito.example`
- Línea internacional ficticia: `+1 555 014 2026`
- Horario: lunes a viernes, de 8:00 a 18:00, hora de Bogotá.  
Lúmina responde reclamaciones simples en un plazo estimado de cinco días hábiles y casos que requieren investigación en un máximo estimado de quince días h
--------------------------------------------------------------------------------
File: lumina_credito_global.md
Relevance: 0.100
lumina_credito_global.md: Lúmina Crédito Global: 16. Canales ficticios de atención

- Sitio web: `https://www.luminacredito.example`
- Correo general: `ayuda@luminacredito.example`
- Correo para pagos: `pagos@luminacredito.example`
- Línea internacional ficticia: `+1 555 014 2026`
- Horario: lunes a viernes, de 8:00 a 18

---

## 7. Creación de la tool de búsqueda

La tool encapsula la búsqueda en OCI Vector Store. Su descripción ayuda al modelo a decidir cuándo utilizarla y el código elimina fragmentos duplicados antes de devolver el contexto.

In [8]:
@tool
def search_knowledge_base(query: str) -> str:
    """Search the configured knowledge base for information relevant to a query."""
    results = client.vector_stores.search(
        vector_store_id=VECTOR_STORE_ID,
        query=query,
        max_num_results=10,
        rewrite_query=False,
    )

    chunks = []
    seen_texts = set()
    for result in results.data:
        for content in result.content:
            text = content.text.strip()
            if not text or text in seen_texts:
                continue

            seen_texts.add(text)
            chunks.append(
                "\n".join(
                    [
                        f"Source: {result.filename}",
                        f"Relevance: {result.score:.3f}",
                        text,
                    ]
                )
            )

    if not chunks:
        return "No results were found in the knowledge base."

    return "\n\n---\n\n".join(chunks)

### Prueba aislada de la tool

Invocamos la tool directamente antes de conectarla al agente. Esto permite validar la recuperación de información por separado.

In [9]:
tool_result = search_knowledge_base.invoke(
    {"query": "¿Cuáles son los requisitos descritos en el documento?"}
)
print(tool_result[:1500])

Source: lumina_credito_global.md
Relevance: 0.140
lumina_credito_global.md: Lúmina Crédito Global: 8. Requisitos generales de aplicación: Para empresas

- Tener al menos 18 meses de operación verificable.
- Registrar ventas anuales mínimas de US$100.000.
- Mantener una cuenta bancaria empresarial.
- Identificar beneficiarios finales y representantes autorizados.
- Presentar información financiera y tributaria vigente.
- Explicar el destino de los fondos.
- Acreditar capacidad de pago mediante flujo de caja.
- Superar controles de cumplimiento, sanciones y prevención de fraude.

lumina_credito_global.md: Lúmina Crédito Global: 9. Proceso de solicitud: Paso 1: Simulación

El interesado selecciona el producto, monto y plazo. La plataforma presenta una cuota ilustrativa basada en una tasa media. La simulación no es una aprobación ni garantiza una tasa.

lumina_credito_global.md: Lúmina Crédito Global: 9. Proceso de solicitud: Paso 2: Formulario

El solicitante proporciona datos personales 

---

## 8. Configuración del modelo para LangChain

`ChatOpenAI` se conecta al endpoint compatible con OpenAI de OCI. El cliente HTTP firma cada solicitud mediante las credenciales IAM.

In [10]:
model = ChatOpenAI(
    model=MODEL_ID,
    api_key="not-used",
    base_url=BASE_URL,
    default_headers={
        "OpenAI-Project": PROJECT_ID,
        "opc-compartment-id": COMPARTMENT_ID,
    },
    http_client=httpx.Client(auth=create_auth()),
    store=False,
    max_retries=2,
)

---

## 9. Creación del agente

El agente combina el modelo y la tool. Las instrucciones le indican que debe consultar la base de conocimiento antes de responder, utilizar el idioma del usuario y reconocer cuando no encuentra información suficiente.

In [11]:
agent = create_agent(
    model=model,
    tools=[search_knowledge_base],
    system_prompt=(
        "You are a knowledge-base assistant. Reply in the same language as "
        "the user. Always call the search_knowledge_base tool before answering "
        "questions about the available documents. Base your answer only on the "
        "retrieved results and clearly say when the knowledge base does not "
        "contain enough information."
    ),
)

### Función auxiliar para hacer preguntas

La función `ask` recibe una pregunta y devuelve únicamente el texto final del agente.

In [12]:
def ask(question: str) -> str:
    """Run the agent and return its final text response."""
    result = agent.invoke(
        {"messages": [{"role": "user", "content": question}]}
    )
    return result["messages"][-1].content

---

## 10. Prueba del agente

La pregunta se realiza en español para comprobar que el agente conserva el idioma del usuario. Puedes reemplazarla por cualquier pregunta relacionada con los documentos indexados.

In [13]:
response = ask(
    "¿Cuáles son los requisitos, costos y condiciones más importantes?"
)
print(response)

A continuación, los requisitos, costos y condiciones más importantes según la base de conocimiento de Lúmina Crédito Global.

Requisitos principales
- Personas:
  - Edad: mínimo 21 años y máximo 70 al finalizar el crédito.
  - Identificación vigente y residencia legal en un país habilitado por Lúmina.
  - Ingresos mensuales brutos desde US$1.500.
  - Cuenta bancaria apta para recibir y enviar dólares.
  - Extractos bancarios de los últimos 3 a 6 meses y comprobantes de ingresos.
  - Autorizar verificaciones de identidad, fraude e historial crediticio.
  - Declarar todas las obligaciones financieras vigentes.
  - Soportes: empleados (certificados laborales y nómina); independientes (contratos, facturas, declaraciones tributarias o reportes de plataformas).
- Empresas (Crédito Pyme Exportadora):
  - Empresa legalmente constituida con ventas internacionales.
  - Documentos: certificado de existencia y representación, identificación tributaria, estados financieros de los últimos 2 años y p

---

## Conclusión

El laboratorio separa los datos de la implementación y permite cambiar documentos, vector stores, regiones y modelos mediante configuración.

Como siguientes pasos puedes:

- Agregar más archivos Markdown a `data/`.
- Seleccionar documentos distintos mediante `OCI_SOURCE_DOCUMENT`.
- Utilizar otro vector store mediante `OCI_VECTOR_STORE_ID`.
- Añadir filtros por atributos durante la búsqueda.
- Incorporar nuevas tools o memoria conversacional.